# Darvas Box

Here's a Python function to create a **Darvas Box** using the most recent **Pivot High** and **Pivot Low**, defined as:

- **Pivot High**: A bar that has at least $n$ lower highs to its **left and right**.
- **Pivot Low**: A bar that has at least $n$ higher lows to its **left and right**.

We will implement this using a pandas DataFrame with OHLC data.

In [ ]:
import pandas as pd


def is_pivot_high(df, i, n):
    if i < n or i + n >= len(df):
        return False
    return all(
        df["High"].iloc[i].item() > df["High"].iloc[i - j].item()
        and df["High"].iloc[i].item() > df["High"].iloc[i + j].item()
        for j in range(1, n + 1)
    )


def is_pivot_low(df, i, n):
    if i < n or i + n >= len(df):
        return False
    return all(
        df["Low"].iloc[i].item() < df["Low"].iloc[i - j].item()
        and df["Low"].iloc[i].item() < df["Low"].iloc[i + j].item()
        for j in range(1, n + 1)
    )


def darvas_box_from_pivots(df, n_pivot=2, box_index=1):
    """
    Returns the nth most recent Darvas Box (1 = most recent).
    """
    pivot_highs = []
    pivot_lows = []

    for i in range(len(df)):
        if is_pivot_high(df, i, n_pivot):
            pivot_highs.append((i, df["High"].iloc[i].item()))
        if is_pivot_low(df, i, n_pivot):
            pivot_lows.append((i, df["Low"].iloc[i].item()))

    # Form Darvas Boxes by matching each pivot high with most recent earlier pivot low
    boxes = []
    for hi_idx, hi_val in reversed(pivot_highs):
        matching_lows = [
            (lo_idx, lo_val) for lo_idx, lo_val in pivot_lows if lo_idx < hi_idx
        ]
        if matching_lows:
            # Get the latest low before this high
            lo_idx, lo_val = max(matching_lows, key=lambda x: x[0])
            boxes.append(
                {
                    "high_index": hi_idx,
                    "low_index": lo_idx,
                    "top": hi_val,
                    "bottom": lo_val,
                    "start": df.index[lo_idx],
                    "end": df.index[hi_idx],
                }
            )

    if len(boxes) < box_index:
        return None  # Not enough boxes found

    return boxes[box_index - 1]

In [ ]:
def pivot_high_series(df, n=3):
    """Plot Pivot High Points"""
    # Create a DataFrame to hold the pivot high markers
    pivots = pd.Series([np.nan] * len(df), index=df.index)

    # Loop through the DataFrame to find pivot highs and mark them
    for i in range(len(df)):
        if is_pivot_high(df, i, n):
            pivots.iloc[i] = df["High"].iloc[i].item()

    return pivots

In [ ]:
def pivot_low_series(df, n=3):
    """Plot Pivot Low Points"""
    # Create a DataFrame to hold the pivot low markers
    pivots = pd.Series([np.nan] * len(df), index=df.index)

    # Loop through the DataFrame to find pivot lows and mark them
    for i in range(len(df)):
        if is_pivot_low(df, i, n):
            pivots.iloc[i] = df["Low"].iloc[i].item()

    return pivots

In [ ]:
def support_resistance_line(df, date, price, extension=5):

    # Find the index
    pivot_index = df.index.get_loc(date)

    # Create a Series for the horizontal line
    line = pd.Series([np.nan] * len(df), index=df.index)

    # Extend the line to the right and 5 days to the left
    left_index = max(0, pivot_index - extension)
    right_index = len(df) - 1  # Extend to the end of the DataFrame
    line.iloc[left_index : right_index + 1] = price

    return line

In [ ]:
import pandas as pd


def darvas_box():
    box = {
        "top": 6059.40,
        "bottom": 5968.61,
        "start": pd.to_datetime("2025-06-11"),
        "end": pd.to_datetime("2025-06-20"),
    }
    return box

In [ ]:
def plot_darvas_box(box):
    # Extract index positions for start and end
    start, end = box["start"], box["end"]
    top, bottom = box["top"], box["bottom"]

    # Create box shading by building a DataFrame with NaNs, except during box range
    total_days = len(df)
    extension_days = int(total_days * 0.1)  # Extend by 10% of total data on each side

    # Find the extended start and end indices
    start_idx = df.index.get_loc(start)
    end_idx = df.index.get_loc(end)

    # Extend start backwards (but not before the beginning of data)
    extended_start_idx = max(0, start_idx - extension_days)
    extended_start = df.index[extended_start_idx]

    # Extend end forwards (but not beyond the end of data)
    extended_end_idx = min(len(df) - 1, end_idx + extension_days)
    extended_end = df.index[extended_end_idx]

    print(f"Original box: {start.date()} to {end.date()}")
    print(f"Extended box: {extended_start.date()} to {extended_end.date()}")

    # Create box shading by building a DataFrame with NaNs, except during extended box range
    top_line = pd.Series([np.nan] * len(df), index=df.index)
    # top_line.loc[start:end] = top
    top_line.loc[extended_start:extended_end] = top

    bottom_line = pd.Series([np.nan] * len(df), index=df.index)
    # bottom_line.loc[start:end] = bottom
    bottom_line.loc[extended_start:extended_end] = bottom

    return top_line, bottom_line

In [ ]:
import yfinance as yf
import numpy as np
import mplfinance as mpf

# Load data
df = yf.download("^GSPC", period="1y", interval="1d", auto_adjust=True)
df.dropna(inplace=True)

# Flatten multi-index columns
if isinstance(df.columns, pd.MultiIndex):
    column_names = ["Date"] + df.columns.get_level_values(0).tolist()
    df = df.reset_index()
    df.columns = column_names
    df = df.set_index("Date")

# Get Darvas Box
# box = darvas_box_from_pivots(df, n_pivot=2, box_index=4)
box = darvas_box()
print(box)

addplots = []
fill_between = []

# Plot
if box:
    top_line, bottom_line = plot_darvas_box(box)

    topline_plot = mpf.make_addplot(top_line, color="blue", width=1.0, linestyle="--")
    bottomline_plot = mpf.make_addplot(
        bottom_line, color="blue", width=1.0, linestyle="--"
    )

    addplots.append(topline_plot)
    addplots.append(bottomline_plot)

    box_fill = dict(y1=top_line.values, y2=bottom_line.values, alpha=0.05, color="blue")

    fill_between.append(box_fill)
else:
    print("No valid Darvas Box found.")

#
# Plot Support & Resistance Lines
#
plot_support_resistance = True

if plot_support_resistance:
    sr1 = support_resistance_line(df, "2025-02-19", 6147.430176)
    sr2 = support_resistance_line(df, "2025-06-11", 6059.399902)
    sr3 = support_resistance_line(df, "2025-05-19", 5968.609863)
    sr4 = support_resistance_line(df, "2025-03-25", 5786.950195)

    curve_sr_1 = mpf.make_addplot(sr1, color="blue", width=1.0, linestyle="--")
    curve_sr_2 = mpf.make_addplot(sr2, color="blue", width=1.0, linestyle="--")
    curve_sr_3 = mpf.make_addplot(sr3, color="blue", width=1.0, linestyle="--")

    addplots.append(curve_sr_1)
    addplots.append(curve_sr_2)
    addplots.append(curve_sr_3)

#
# Plot Pivot Points
#
plot_pivot_points = True

if plot_pivot_points:
    pivot_high_markers = pivot_high_series(df, 5)
    pivot_low_markers = pivot_low_series(df, 5)

    filtered_highs = pivot_high_markers.dropna()
    filtered_lows = pivot_low_markers.dropna()
    # print(filtered_highs)
    # print(filtered_lows)

    highs_plot = mpf.make_addplot(
        pivot_high_markers, type="scatter", markersize=10, marker="o", color="blue"
    )
    lows_plot = mpf.make_addplot(
        pivot_low_markers, type="scatter", markersize=10, marker="o", color="red"
    )

    addplots.append(highs_plot)
    addplots.append(lows_plot)


# Plot with mplfinance
mpf.plot(
    df,
    type="ohlc",
    style="charles",
    addplot=addplots,
    fill_between=fill_between,
    title=f"Darvas Box",
    volume=True,
    figscale=1.2,
    figratio=(16, 9),
)

## Pivot Highs and Lows

In [ ]:
pivot_highs = pivot_high_series(df).dropna()
pivot_highs

In [ ]:
pivot_lows = pivot_low_series(df).dropna()
pivot_lows